In [10]:
pip install -q pinecone sentence-transformers pandas


Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
import time
import json
import pandas as pd
from pathlib import Path
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

pinecone_api_key = os.getenv("PINECONE_API_KEY", "pcsk_6z2qwH_MAexzdkdHPXK2FdmsZwzKunmp3ejfrTUKjeACrUb297LH6gKG1fk1iSnEVm5FnE")
if not pinecone_api_key:
    raise ValueError("Set PINECONE_API_KEY in your environment before running this notebook.")

pc = Pinecone(api_key=pinecone_api_key)


In [24]:
notebook_dir = Path.cwd()
data_dir = notebook_dir / 'retail_data'
if not data_dir.exists():
    data_dir = notebook_dir / 'Retail-Data--main' / 'retail_data'
if not data_dir.exists():
    raise FileNotFoundError(f"retail_data directory not found under {notebook_dir} or {notebook_dir / 'Retail-Data--main'}")

print('Using data directory:', data_dir)
print('Category folders:', sorted([p.name for p in data_dir.iterdir() if p.is_dir()]))


Using data directory: c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinecone\Retail-Data--main\Retail-Data--main\retail_data
Category folders: ['Apparel', 'Electronics', 'Health_&_Beauty', 'Home_&_Kitchen', 'Sports_&_Outdoors', '__pycache__']


In [25]:
def load_retail_json(data_dir: Path) -> pd.DataFrame:
    rows = []
    for json_file in sorted(data_dir.rglob('*.json')):
        try:
            with json_file.open('r', encoding='utf-8') as f:
                data = json.load(f)

            text = data.get('fullText') or data.get('description') or ''
            if not text.strip():
                continue

            rows.append({
                'category': data.get('category') or json_file.parent.name,
                'file_id': data.get('fileId') or json_file.stem,
                'file_name': json_file.name,
                'file_path': str(json_file),
                'title': data.get('title'),
                'description': data.get('description'),
                'text': text
            })

        except Exception as e:
            print('Failed to load', json_file, e)

    return pd.DataFrame(rows)

df = load_retail_json(data_dir)
print('Loaded JSON rows:', len(df))
df.head()


Loaded JSON rows: 500


,category,file_id,file_name,file_path,title,description,text
0,Apparel,APP-001,apparel_001.json,c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinec...,Apparel Retail Insight 1,A comprehensive informational overview for app...,"Fashion apparel, seasonal collections, fabrics..."
1,Apparel,APP-002,apparel_002.json,c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinec...,Apparel Retail Insight 2,A comprehensive informational overview for app...,"Fashion apparel, seasonal collections, fabrics..."
2,Apparel,APP-003,apparel_003.json,c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinec...,Apparel Retail Insight 3,A comprehensive informational overview for app...,"Fashion apparel, seasonal collections, fabrics..."
3,Apparel,APP-004,apparel_004.json,c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinec...,Apparel Retail Insight 4,A comprehensive informational overview for app...,"Fashion apparel, seasonal collections, fabrics..."
4,Apparel,APP-005,apparel_005.json,c:\Users\DAYA RAM YADAV\OneDrive\Desktop\Pinec...,Apparel Retail Insight 5,A comprehensive informational overview for app...,"Fashion apparel, seasonal collections, fabrics..."


In [26]:
try:
    index_name = 'retail-json-384'
    namespace = 'retail_json'
    embedding_dimension = 384

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=embedding_dimension,
            metric='cosine',
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
        time.sleep(10)

    index = pc.Index(index_name)

    vectors = []
    batch_size = 100
    chunk_size = 500
    vector_id = 1

    for _, row in df.iterrows():
        text = str(row['text'])
        if not text.strip():
            continue

        for start in range(0, len(text), chunk_size):
            chunk = text[start:start + chunk_size].strip()
            if not chunk:
                continue

            embedding = model.encode(chunk).tolist()
            vectors.append({
                'id': f'{row['file_id']}_chunk_{vector_id}',
                'values': embedding,
                'metadata': {
                    'category': row['category'],
                    'file_name': row['file_name'],
                    'title': row['title'],
                    'description': row['description'],
                    'text': chunk
                }
            })
            vector_id += 1

            if len(vectors) >= batch_size:
                index.upsert(vectors=vectors, namespace=namespace)
                vectors = []

        print('Queued for upload:', row['file_name'])

    if vectors:
        index.upsert(vectors=vectors, namespace=namespace)

    print('Upload completed.')
    print(index.describe_index_stats())

except Exception as e:
    if '401' in str(e) or 'Unauthorized' in str(e):
        print('ERROR: Invalid or missing Pinecone API key.')
        print('Please set PINECONE_API_KEY environment variable with your actual API key.')
        print(f'Current value: {pinecone_api_key[:20]}...' if pinecone_api_key else 'Not set')
    else:
        print(f'ERROR: {e}')
        raise


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Queued for upload: apparel_001.json
Queued for upload: apparel_002.json
Queued for upload: apparel_003.json
Queued for upload: apparel_004.json
Queued for upload: apparel_005.json
Queued for upload: apparel_006.json
Queued for upload: apparel_007.json
Queued for upload: apparel_008.json
Queued for upload: apparel_009.json
Queued for upload: apparel_010.json
Queued for upload: apparel_011.json
Queued for upload: apparel_012.json
Queued for upload: apparel_013.json
Queued for upload: apparel_014.json
Queued for upload: apparel_015.json
Queued for upload: apparel_016.json
Queued for upload: apparel_017.json
Queued for upload: apparel_018.json
Queued for upload: apparel_019.json
Queued for upload: apparel_020.json
Queued for upload: apparel_021.json
Queued for upload: apparel_022.json
Queued for upload: apparel_023.json
Queued for upload: apparel_024.json
Queued for upload: apparel_025.json
Queued for upload: apparel_026.json
Queued for upload: apparel_027.json
Queued for upload: apparel_0

In [27]:
def query_retail(question: str, top_k: int = 3):
    query_embedding = model.encode(question).tolist()
    return index.query(
        vector=query_embedding,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )

question = input('Enter your question: ')
results = query_retail(question, top_k=3)

print('\nTop results')
print('=' * 100)
for rank, match in enumerate(results.matches, start=1):
    print(f'Rank: {rank}')
    print(f'Score: {match.score:.4f}')
    print(f'Category: {match.metadata.get("category")}')
    print(f'Title: {match.metadata.get("title")}')
    print(f'File: {match.metadata.get("file_name")}')
    print('Content:')
    print(match.metadata.get("text"))
    print('=' * 100)



Top results
Rank: 1
Score: 0.3273
Category: Apparel
Title: Apparel Retail Insight 4
File: apparel_004.json
Content:
Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and store merchandising. This document offers a fulltext explanation of retail strategy, sales optimization, inventory management, customer service, and rich product storytelling for apparel. It includes industry best practices, merchandising recommendations, performance metrics, supplier coordination, digital fulfillment, and examples of how brands can improve customer retention. Each narrative touches on the current reta
Rank: 2
Score: 0.3273
Category: Apparel
Title: Apparel Retail Insight 55
File: apparel_055.json
Content:
Fashion apparel, seasonal collections, fabrics, sizing, trend forecasting and store merchandising. This document offers a fulltext explanation of retail strategy, sales optimization, inventory management, customer service, and rich product storytelling for apparel. It includes